In [0]:
%run /Users/malkeet21.uk@gmail.com/Databricks-Data-Engineering-Assesment/common_utils/utils

In [0]:
from pyspark.sql.functions import col

df_bronze = spark.table(BRONZE_TABLE)

# Step 1: rename _c2.._c20 using the labels on the header row ("90 & over" -> "90_and_over")
df_step1 = rename_from_header_row(df_bronze, "_c2", HEADER_MARKER, df_bronze.columns[2:21])

# Step 2: rename _c0 and _c1
df_step2 = df_step1.withColumnRenamed("_c0", "population").withColumnRenamed("_c1", "year")

# Step 3: remove the bottom 7 rows (notes)
df_step3 = drop_last_rows(df_step2, NOTE_ROW_COUNT)

#display(df_step3)

# Step 4: fill population down, keep only data rows, drop audit columns
df_step4 = (
    fill_down(df_step3, "population")
    .filter(col("year").isNotNull())                          # title, header and blank rows have no year
    .drop("_source_row_id", "_source_file", "_ingested_at")
)

df_persons = df_step4.filter(col("population") == "Persons").orderBy("year")
df_males = df_step4.filter(col("population") == "Males").orderBy("year")
df_females = df_step4.filter(col("population") == "Females").orderBy("year")

# Step 5: display Persons
display(df_persons)


In [0]:
# Save to silver (gold reads these tables)
write_delta_table(df_persons, SILVER_PERSONS_TABLE)
write_delta_table(df_males, SILVER_MALES_TABLE)
write_delta_table(df_females, SILVER_FEMALES_TABLE)